# Anatomy of a Prompt — What the Model Actually Sees

In [ ]:
# If you are running this on Google Colab, uncomment and run the line below first.
# !pip install -q anthropic tiktoken

## The same question, two completely different answers

Try asking a model to "fix this code" and it makes conservative, minimal changes. Ask it to "rewrite this code so it's clean and production-ready" and it completely restructures everything. The code you handed it was identical. The only difference was a handful of words.

This is not a quirk — it is the fundamental nature of how language models work. The model cannot see your intent. It cannot ask follow-up questions. It only has access to the exact sequence of tokens you gave it, and every inference it makes starts from that sequence.

Small wording changes flip the output because the model is doing next-token prediction over your exact words. Change the words, change the prediction.

Before you can get reliable, useful output from any LLM, you need to understand what a prompt actually is under the hood — not as text you type into a chat box, but as a structured data structure the model processes.

## The message format — how prompts are actually structured

When you use a chat-based model like Claude or GPT, you are not sending a single blob of text. You are sending a **list of messages**, where each message has two fields:

- `role` — who is speaking: `system`, `user`, or `assistant`
- `content` — what they said

**The analogy:** think of it like a film script. There is a director's note at the top (the system prompt), then the script alternates between the audience asking questions (user) and the actor responding (assistant). The model reads the entire script up to the current line and predicts what the actor says next.

Here is what the raw API call looks like:

In [ ]:
import json

# This is what every chat API call looks like under the hood
messages = [
    {"role": "system",    "content": "You are a concise assistant. Answer in one sentence."},
    {"role": "user",      "content": "What is gradient descent?"},
]

print("What you send to the API:")
print(json.dumps(messages, indent=2))
print()
print("Three roles exist: 'system', 'user', 'assistant'")
print("The model reads them all in order before predicting the next token.")

## The three roles — what each one does

Each role carries different weight and serves a different purpose. Understanding this is what separates prompts that work from prompts that fight the model.

**`system`** — the director's note before filming starts. It sets the model's persona, constraints, output format, and anything that should stay constant across the entire conversation. The model treats this as background truth. Most API clients send it once at the start.

**`user`** — the actual input from the human. This is the question, task, document to process, or instruction you want acted on. It can be short ("Summarise this") or very long (an entire codebase).

**`assistant`** — the model's own prior responses, injected back into the conversation. This is how multi-turn conversations work — the model sees its own history as part of the context and can build on it, reference it, or correct it.

**The analogy:** system is a job description handed to an employee on day one. user is the task assigned each morning. assistant is the employee's notes from yesterday — their memory of the work already done.

In [ ]:
import json

# A multi-turn conversation showing all three roles
conversation = [
    {
        "role": "system",
        "content": "You are a Python tutor. Keep explanations short and always include a one-line code example."
    },
    {
        "role": "user",
        "content": "What is a list comprehension?"
    },
    {
        "role": "assistant",
        "content": "A list comprehension builds a list in a single expression. Example: `squares = [x**2 for x in range(5)]` gives [0, 1, 4, 9, 16]."
    },
    {
        "role": "user",
        "content": "Can I add a filter to that?"
    },
    # The model will now respond here, with full memory of the system prompt
    # and the previous exchange
]

print("Full conversation context the model receives:")
for msg in conversation:
    role_label = f"[{msg['role'].upper():<10}]"
    print(f"{role_label}  {msg['content']}")

print()
print("The model predicts the next [ASSISTANT] turn using all of the above.")
print("It knows the system persona, the previous question, and its own prior answer.")

## The context window — a sliding spotlight, not infinite memory

The model does not have persistent memory between API calls. It only knows what is in the current request. The **context window** is the maximum number of tokens the model can process at once — system prompt, conversation history, documents, and your current question all have to fit inside it.

**The analogy:** imagine a person who can only read a fixed-length scroll. You can hand them as much text as fits on the scroll. When the scroll fills up, something has to fall off the end to make room for the new text. The model only reasons over what is currently on the scroll.

Modern models have large context windows — Claude supports up to 200K tokens, some models go higher. But there is a real phenomenon called **lost in the middle**: studies show models pay more attention to content at the very beginning and very end of a long context, and can miss information buried in the middle. Position matters even within a large window.

In [ ]:
import tiktoken

# Use the cl100k_base encoding (GPT-4 / Claude-compatible approximation)
encoder = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(encoder.encode(text))

system_prompt = "You are a helpful assistant. Be concise and accurate."

user_messages = [
    "What is a transformer model?",
    "Summarise the following article: " + ("Neural networks learn by adjusting weights. " * 50),
    "What year was Python created?",
]

system_tokens = count_tokens(system_prompt)
print(f"System prompt       : {system_tokens:>5} tokens")
print(f"{'Message':<35}  Tokens")
print("-" * 48)

total = system_tokens
for msg in user_messages:
    t = count_tokens(msg)
    total += t
    label = msg[:35] + "..." if len(msg) > 35 else msg
    print(f"{label:<38}  {t:>5}")

print("-" * 48)
print(f"{'Total context size':<38}  {total:>5} tokens")
print()
print(f"Claude's context window: ~200,000 tokens")
print(f"This conversation uses {total/200_000*100:.2f}% of it.")
print("Long documents, large codebases, or many conversation turns fill this fast.")

## Why small wording changes flip the output

The model predicts one token at a time, conditioned on everything that came before. That means early words in your prompt constrain what's probable later. Change a word early in the prompt, and you shift the probability distribution over every subsequent token.

**The analogy:** think about autocomplete on your phone. Type "I want to" and your phone suggests "eat", "buy", "go". Type "I need to" and it suggests "sleep", "tell", "check". The model is doing the same thing, but at the scale of entire paragraphs. Your first few words load a prior that shapes everything else.

Let's build a side-by-side comparison to make this concrete. We'll show the same intent phrased two different ways and annotate what the model is likely to do differently.

In [ ]:
import textwrap

prompt_pairs = [
    {
        "intent": "Get help fixing buggy code",
        "weak":   "Fix this code: def add(a,b): return a-b",
        "strong": "This function should return the sum of two numbers but it subtracts instead. Fix the bug and explain what was wrong: def add(a,b): return a-b",
        "why":    "'Fix' is ambiguous. The strong version names the expected behaviour, the actual behaviour, and asks for an explanation — constraining the model toward exactly what you want.",
    },
    {
        "intent": "Get a brief explanation",
        "weak":   "Tell me about neural networks.",
        "strong": "In two sentences, explain what a neural network is to someone who understands basic statistics but has never studied machine learning.",
        "why":    "Without length and audience constraints, the model defaults to a thorough textbook answer. The strong version gives it a length target and a specific reader — it knows exactly what register and depth to use.",
    },
    {
        "intent": "Brainstorm product names",
        "weak":   "Give me names for my app.",
        "strong": "Generate 5 short, memorable names for a productivity app aimed at freelance developers. Each name should be one or two words and easy to pronounce.",
        "why":    "The weak version produces generic names. The strong version supplies audience, count, length, and pronounceability — each constraint prunes a large slice of the output space.",
    },
]

wrap = lambda t: '\n'.join(textwrap.wrap(t, width=55))

for i, pair in enumerate(prompt_pairs, 1):
    print(f"Example {i}: {pair['intent']}")
    print(f"  WEAK  : {pair['weak']}")
    print(f"  STRONG: {wrap(pair['strong'])}")
    print(f"  WHY   : {wrap(pair['why'])}")
    print()

## The system prompt — your most powerful lever

Out of all parts of the prompt, the system message has the most persistent influence. It sets the model's persona before any user input arrives, and stays active for the entire conversation.

The system prompt is where you define: who the model is, what it should and should not do, what format it should use, and what tone it should maintain. Think of it as configuration, not instructions.

**The analogy:** before a customer support call centre opens for the day, the manager briefs the agents — here is the product, here is what you can offer, here is how we talk to customers. Every call that day happens within that framing. The system prompt is that morning briefing.

In [ ]:
import json

# Three different system prompts for the exact same user question
user_question = "What should I know about Python decorators?"

system_prompts = {
    "No system prompt": None,
    "Concise tutor":    "You are a Python tutor. Explain concepts clearly in 2-3 sentences max. Use a one-line code example.",
    "Senior reviewer":  "You are a senior engineer reviewing code. Be direct and opinionated. Skip basic definitions and focus on practical traps and non-obvious behaviour.",
    "ELI5 mode":        "Explain everything as if the reader is twelve years old with no programming experience. Use everyday analogies. Never use technical jargon without explaining it first.",
}

print(f"User question: '{user_question}'")
print("=" * 60)
print()

for label, system in system_prompts.items():
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_question})

    print(f"System: [{label}]")
    if system:
        print(f"  '{system[:80]}{'...' if len(system) > 80 else ""}'")
    else:
        print("  (none — model uses its default behaviour)")
    print(f"  Message count: {len(messages)}")
    print()

print("Same question. Four completely different responses in production.")
print("The system prompt is doing all the work.")

## Putting it together — a live API call with all the pieces

Now let's wire everything together into one real API call. This is the actual pattern you will use in every application: a system prompt that sets the persona, a user message with the task, and reading back the response.

In [ ]:
import os
import anthropic
import tiktoken

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
encoder = tiktoken.get_encoding("cl100k_base")

system_prompt = """You are a concise technical explainer. 
When asked to compare two things, respond with exactly three bullet points per item. 
Be direct and skip any introductory sentences."""

user_message = "Compare synchronous and asynchronous programming."

input_tokens = len(encoder.encode(system_prompt + user_message))

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    system=system_prompt,
    messages=[{"role": "user", "content": user_message}],
)

output_text   = response.content[0].text
output_tokens = response.usage.output_tokens

print(f"System prompt : {len(encoder.encode(system_prompt)):>4} tokens")
print(f"User message  : {len(encoder.encode(user_message)):>4} tokens")
print(f"Input total   : {input_tokens:>4} tokens")
print(f"Output        : {output_tokens:>4} tokens")
print()
print("Response:")
print("-" * 50)
print(output_text)

## The framing effect — same fact, opposite interpretation

One of the most important things to understand about prompting is that the model does not retrieve facts from a database — it generates text that is statistically consistent with your prompt. That means framing the prompt differently can produce opposite conclusions from the same underlying information.

This is not a bug. It is a direct consequence of how the model works, and it is why prompting discipline matters. Let's make the framing effect visible.

In [ ]:
import os
import anthropic

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

topic = "using microservices architecture for a new startup project"

framing_prompts = [
    ("Neutral",    f"List three considerations about {topic}."),
    ("Pro-biased",  f"What are three compelling reasons to use {topic}?"),
    ("Anti-biased", f"What are three serious risks of {topic} that engineers often overlook?"),
]

system = "You are a senior software architect. Be direct and specific. No preamble."

for label, prompt in framing_prompts:
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    print(f"[{label} framing]")
    print(f"Prompt: {prompt}")
    print("Response:")
    print(response.content[0].text)
    print("-" * 55)
    print()

print("The system persona is identical. The topic is identical.")
print("Only the framing changed — but the response profile is completely different.")

## Visualising the anatomy of a prompt

Let's make the structure of a real prompt visible by breaking it into its parts and measuring each one. This builds intuition for how a prompt's token budget is actually spent before you hit the API.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tiktoken

encoder = tiktoken.get_encoding("cl100k_base")

prompt_parts = {
    "System prompt\n(persona & rules)": """
        You are a senior data engineer. Be concise. 
        When writing SQL, always use CTEs over subqueries. 
        Format output as markdown code blocks.
    """.strip(),
    "Conversation\nhistory (2 turns)": """
        User: How do I count distinct users per day?
        Assistant: Use DATE_TRUNC and COUNT(DISTINCT user_id) grouped by day.
        User: Can you show me with a CTE?
        Assistant: WITH daily_users AS (SELECT DATE_TRUNC('day', created_at) AS day, COUNT(DISTINCT user_id) AS users FROM events GROUP BY 1) SELECT * FROM daily_users ORDER BY day;
    """.strip(),
    "Retrieved context\n(RAG document)": """
        The events table has columns: event_id, user_id, event_type, created_at, session_id.
        Approximate row count: 450 million. Partitioned by created_at (daily). 
        Indexed on: user_id, event_type, created_at.
    """.strip(),
    "Current user\nmessage": "Now write a query that shows the top 10 users by number of distinct sessions in the last 30 days.",
}

token_counts = {label: len(encoder.encode(text)) for label, text in prompt_parts.items()}
total = sum(token_counts.values())

colours = ["steelblue", "#5b9bd5", "seagreen", "tomato"]
labels  = list(token_counts.keys())
counts  = list(token_counts.values())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Horizontal stacked bar showing context budget
left = 0
for i, (label, count) in enumerate(token_counts.items()):
    ax1.barh(0, count, left=left, color=colours[i], height=0.5, label=label)
    if count > 15:
        ax1.text(left + count / 2, 0, str(count), ha='center', va='center',
                 fontsize=9, color='white', fontweight='bold')
    left += count

ax1.set_xlim(0, total * 1.05)
ax1.set_yticks([])
ax1.set_xlabel("Tokens")
ax1.set_title(f"Context window usage — {total} tokens total")
ax1.legend(loc='lower right', fontsize=8)

# Pie chart showing proportion
wedges, texts, autotexts = ax2.pie(
    counts, labels=None, colors=colours,
    autopct='%1.0f%%', startangle=140,
    pctdistance=0.75, wedgeprops=dict(edgecolor='white')
)
ax2.set_title("Proportion of context by part")
patches = [mpatches.Patch(color=colours[i], label=labels[i]) for i in range(len(labels))]
ax2.legend(handles=patches, loc='lower center', bbox_to_anchor=(0.5, -0.2), fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

print(f"Total context: {total} tokens out of ~200,000 available (Claude).")
print("In real RAG systems, retrieved context often dominates the token budget.")
print("Notice how the current user message is the smallest part — framing it well matters a lot.")

## Key takeaways

- **A prompt is a list of messages**, each with a `role` and `content` — not a single string. The model reads all of them before generating.
- **Three roles**: `system` sets the persona (persists across the conversation), `user` carries the input, `assistant` carries prior responses the model can reference.
- **The context window** is the total token budget — system prompt, history, documents, and the current message all consume it. There is no persistent memory outside it.
- **Small wording changes flip the output** because the model conditions every next token on everything that came before. Early words load a prior that shapes the whole response.
- **The system prompt is your most powerful lever** — it sets the model's behaviour before any user input arrives and is worth spending real time on.
- **Framing is not neutral** — asking "what are the risks" versus "what are the benefits" produces structurally opposite responses from the same model on the same topic.
- **Position in context matters**: content at the beginning and end of a long context gets more attention than content buried in the middle.

---

Next up: **Zero-Shot Prompting** — now that you know what the model sees, let's explore how far you can get with a well-structured prompt and no examples at all.